        # 03 現在価値と感度分析（RQ4）

        - RQ4：**生涯所得の最大化と、現在価値（NPV）の最大化は同じか**
        - 感度分析：**どの仮定が結果を大きく左右するか**

> 架空のデータによる**教育用シミュレーション**です。実際の賃金統計ではなく、将来の所得の予測でもありません。金額は**万円**、税・社会保険料を引く前の**額面**です。

## 準備：実験環境（最初に必ず実行）

F1 リポジトリの `.py`（`scenarios.py`・`income_analysis.py`・`optimization.py` など）を import できるように、作業ディレクトリ（F1 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F1 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

> `textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと `optimization.py` の import が失敗します。

In [ ]:
#@title 準備（F1 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                  # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f1-career-income-optimization"  # リポジトリの中の F1 フォルダ（ルートが F1 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f1_root(p):
    return (Path(p) / "career_model.py").exists() and (Path(p) / "optimization.py").exists()


def find_f1_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F1 のルート
        if is_f1_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f1_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F1 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f1_upload")
    for hit in glob.glob("/content/f1_upload/**/career_model.py", recursive=True):
        if is_f1_root(Path(hit).parent):
            return Path(hit).parent
    return None


F1_ROOT = find_f1_root()
if F1_ROOT is None:
    raise RuntimeError("F1 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F1_ROOT)
if str(F1_ROOT) not in sys.path:
    sys.path.insert(0, str(F1_ROOT))
if not (F1_ROOT / "textbook" / "max.py").exists():
    print("⚠️ textbook/ がありません。optimization.py の import が失敗します。")

# ---- 既存の .py（計算の本体）と、発表用のグラフ ----
import matplotlib.pyplot as plt
import career_model as cm
import scenarios as sc
import income_analysis as ia
import optimization as op
import presentation_charts as pc

pc.setup_style()                      # 発表用に文字を大きめにする（pc.setup_style(1.3) でさらに大きく）
careers, settings = sc.load_scenarios()
named = {s["name"]: s for s in careers}
RATE = settings.get("discount_rate", ia.DEFAULT_DISCOUNT_RATE)

SAVE_FIGURES = False                  # True にすると、スライド用の PNG を notebooks/figures/ に保存する
FIG_DIR = "notebooks/figures"

def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", pc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()

print("F1_ROOT:", F1_ROOT)
print("キャリア:", [s["label"] for s in careers], " 割引率:", RATE)

## 3 つの評価基準

| 指標 | 定義 |
| --- | --- |
| 生涯所得 | 18〜64 歳の額面の年収の合計 |
| 累積純収入 | 生涯所得 − キャリア関連費用（進学・資格・転職）※手取りではない |
| NPV | $\displaystyle \sum_t \frac{\text{CashFlow}_t}{(1+r)^t}$、$t$ = 年齢 − 18 |

割引率 $r$ が高いほど、**遠い将来のお金の価値が小さく**なります。

In [ ]:
results = [ia.evaluate(s, RATE) for s in careers]
print(f"{'Scenario':16}{'Lifetime Income':>17}{'Costs':>8}{'Net Income':>12}{'NPV (r=' + format(RATE, '.0%') + ')':>13}")
for r in results:
    print(f"{r['label']:16}{r['gross']:>17,.0f}{r['costs']:>8,.0f}{r['net']:>12,.0f}{r['npv']:>13,.0f}")
print()
for key, name in op.CRITERIA.items():
    print(f"{name:22}: " + " > ".join(r["label"] for r in op.rank(results, key)))

## 生涯所得と NPV の比較

In [ ]:
#@title グラフを表示（コードは隠しています）
show(pc.plot_metric_bars(results, "gross", "Lifetime gross income (no discounting)"), "rq4_gross_bars")
show(pc.plot_metric_bars(results, "npv", f"NPV at age 18 (discount rate {RATE:.0%})"), "rq4_npv_bars")

## 割引率を変えると、NPV はどう変わるか

In [ ]:
#@title グラフを表示（コードは隠しています）
rates = [i / 1000 for i in range(0, 61, 5)]      # 0%〜6%（0.5% 刻み）
show(pc.plot_npv_vs_rate(careers, rates, "NPV vs discount rate"), "rq4_npv_vs_rate")

## 割引率と順位：線が交差すると順位が入れ替わる

In [ ]:
#@title グラフを表示（コードは隠しています）
show(pc.plot_rank_by_rate(careers, [0.0, 0.01, 0.02, 0.03, 0.04, 0.05]), "rq4_rank_by_rate")
for r in [0.0, 0.01, 0.03, 0.05]:
    print(f"r = {r:.0%}: " + " > ".join(x["label"] for x in op.rank([ia.evaluate(s, r) for s in careers], "npv")))

## なぜ順位が入れ替わるのか

- 進学は「**最初に**費用と所得 0 の期間があり、**後で**高い年収」→ 割引率が高いと、後の高い年収の価値が小さくなる
- 早く就職するキャリアは「**最初から**所得がある」→ 割引の影響を受けにくい
- 割引率 0% の NPV は累積純収入と同じ（テストで確認済み）

→ **何を最大化するか（評価基準）で「最適な選択」は変わる**

## 感度分析：University と Early Job の差を左右するのは？

University の仮定を 1 つずつ ±20% 変え、累積純収入の差（University − Early Job）を比べます。

In [ ]:
base, uni = named["early_job"], named["university"]

def diff(s):
    return ia.cumulative_net_income(s) - ia.cumulative_net_income(base)

reference = diff(uni)
sens = []
for key, label in [("initial_salary", "initial salary"), ("annual_growth_rate", "growth rate"),
                   ("education_cost", "education cost")]:
    low = diff(sc.with_changes(uni, **{key: uni[key] * 0.8}))
    high = diff(sc.with_changes(uni, **{key: uni[key] * 1.2}))
    sens.append({"label": f"{label} ±20%", "low": low, "high": high})
show(pc.plot_tornado(sens, reference, "Sensitivity: University minus Early Job (cumulative net income)",
                     "difference [10k JPY]"), "rq4_tornado")
print(f"基準の差：{reference:,.0f} 万円")
for r in sens:
    print(f"{r['label']:22}: {r['low']:,.0f} 〜 {r['high']:,.0f} 万円")

## 自分たちの感度分析

In [ ]:
# TODO：変える仮定・変える幅（±20%）・比べる 2 つのキャリアを選んで再実行しよう
# 例：Early Job 側の昇給率、資格取得費用、転職費用、割引率（NPV で比べる場合）
my_target, my_base = named["certificate"], named["early_job"]
print(f"Certificate − Early Job（累積純収入）：{ia.cumulative_net_income(my_target) - ia.cumulative_net_income(my_base):,.0f} 万円")

## 考察（チームで記入）

- **TODO**：RQ4 への答え：生涯所得の最大化と NPV の最大化は同じか。違うのはどんなときか
- **TODO**：割引率はいくつにするのが妥当か（自分にとっての「今のお金」と「将来のお金」の価値）
- **TODO**：結果を最も大きく左右する仮定はどれか。その仮定の値をどう決めればよいか

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索、第6章 挿入ソート）
- F1 リポジトリの `README.md`（モデルの仮定、計算量、モデルの限界と倫理的配慮）
- データ：`data/sample_careers.json`（この教材のための**架空のデータ**）